In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import re
import pandas as pd
import numpy as np
import mne
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report

folder = Path("/content/drive/MyDrive/eeg-during-mental-arithmetic-tasks-1.0.0")
output_folder = Path("/content/drive/MyDrive/output-mental-arithmetic")

eeg_channels = [
    "Fp1", "Fp2", "F3", "F4", "F7", "F8", "T3", "T4", "C3", "C4",
    "T5", "T6", "P3", "P4", "O1", "O2", "Fz", "Cz", "Pz",
]

bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 45),
}

sfreq = 500
chunk_size = 500
LABEL_MODE = "state"

def process_eeg_file(input_file):
    output_file = output_folder / f"{input_file.stem}-processed.csv"
    if output_file.exists():
        print(f"Already processed, skipping: {input_file.name}")
        return pd.read_csv(output_file)

    raw = mne.io.read_raw_edf(str(input_file), preload=True, verbose=False)
    raw.rename_channels({ch: ch.replace("EEG ", "").replace("ECG ", "").strip() for ch in raw.ch_names})

    missing = [ch for ch in eeg_channels if ch not in raw.ch_names]
    if missing:
        print(f"Skipping {input_file.name}, missing channels: {missing}")
        return None

    raw.pick(eeg_channels)
    raw.reorder_channels(eeg_channels)

    if raw.info["sfreq"] != sfreq:
        raw.resample(sfreq, verbose=False)

    processed_columns = {}
    for band_name, (f_min, f_max) in bands.items():
        raw_band = raw.copy()
        raw_band.filter(l_freq=f_min, h_freq=f_max, fir_design="firwin", verbose=False)
        band_data = raw_band.get_data()

        number_of_chunks = band_data.shape[1] // chunk_size
        usable_samples = number_of_chunks * chunk_size
        band_data = band_data[:, :usable_samples]

        band_chunks = band_data.reshape(len(eeg_channels), number_of_chunks, chunk_size)
        chunk_intensities = np.mean(band_chunks ** 2, axis=2)
        chunk_intensities = np.log10(chunk_intensities + 1e-20)

        for channel_index, channel_name in enumerate(eeg_channels):
            processed_columns[f"{channel_name}_{band_name}"] = chunk_intensities[channel_index]

    processed_df = pd.DataFrame(processed_columns)
    processed_df.insert(0, "time_seconds", np.arange(len(processed_df)))

    output_folder.mkdir(exist_ok=True, parents=True)
    processed_df.to_csv(output_file, index=False)
    print("Processed:", input_file.name)
    print("Saved as:", output_file.name)
    print("Shape:", processed_df.shape)
    return processed_df

def process_all_eeg_files():
    raw_files = sorted(folder.glob("Subject[0-9][0-9]_[12].edf"))
    print(f"Found {len(raw_files)} EDF files matching the pattern in {folder}")
    if not raw_files:
        print("^ that's 0 files - check your Google Drive folder structure.")
        return

    processed_count = 0
    for input_file in raw_files:
        result = process_eeg_file(input_file)
        if result is not None:
            processed_count += 1
    print(f"\nProcessed {processed_count} / {len(raw_files)} files successfully.")

# Run processing pipeline
process_all_eeg_files()

FEATURE_COLUMNS = [f"{ch}_{band}" for ch in eeg_channels for band in bands.keys()]

def label_from_filename(filename):
    if LABEL_MODE == "state":
        match = re.search(r"Subject\d+_([12])", filename)
        return {"1": "rest", "2": "arithmetic"}[match.group(1)]
    match = re.search(r"(Subject\d+)_", filename)
    return match.group(1).lower()

def subject_from_filename(filename):
    match = re.search(r"(Subject\d+)_", filename)
    return match.group(1)

feature_frames = []
groups = []
subjects = []
labels = []

processed_csvs = sorted(output_folder.glob("*-processed.csv"))
print(f"\nFound {len(processed_csvs)} processed CSVs in {output_folder}")

for file in processed_csvs:
    df = pd.read_csv(file)
    missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing:
        print(f"Skipping {file.name}, missing columns: {missing}")
        continue

    label = label_from_filename(file.name)
    subject = subject_from_filename(file.name)

    feature_frames.append(df[FEATURE_COLUMNS])
    groups.extend([file.stem] * len(df))
    subjects.extend([subject] * len(df))
    labels.extend([label] * len(df))

if not feature_frames:
    raise RuntimeError(
        "No usable feature rows were collected. Verify that the files were generated in Drive."
    )

X = pd.concat(feature_frames, ignore_index=True)
y = np.array(labels)
groups = np.array(groups)
subjects = np.array(subjects)

n_nan = X.isna().sum().sum()
if n_nan:
    print(f"WARNING: {n_nan} NaN values found in features - replacing with 0.")
    X = X.fillna(0.0)

print("\nTotal samples:", X.shape[0])
print("Classifying by:", LABEL_MODE)
print("Classes:", sorted(set(y)))
print(pd.Series(y).value_counts())

if len(set(y)) < 2:
    raise RuntimeError(f"Only one class present in the labels ({set(y)}) - can't train.")

X_norm = X.copy()
for subject in np.unique(subjects):
    mask = subjects == subject
    mu = X.loc[mask].mean()
    sigma = X.loc[mask].std().replace(0, 1.0)
    X_norm.loc[mask] = (X.loc[mask] - mu) / sigma
X = X_norm.to_numpy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=18,
    min_samples_leaf=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_test))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 72 EDF files matching the pattern in /content/drive/MyDrive/eeg-during-mental-arithmetic-tasks-1.0.0
Processed: Subject00_1.edf
Saved as: Subject00_1-processed.csv
Shape: (182, 96)
Processed: Subject00_2.edf
Saved as: Subject00_2-processed.csv
Shape: (62, 96)
Processed: Subject01_1.edf
Saved as: Subject01_1-processed.csv
Shape: (182, 96)
Processed: Subject01_2.edf
Saved as: Subject01_2-processed.csv
Shape: (62, 96)
Processed: Subject02_1.edf
Saved as: Subject02_1-processed.csv
Shape: (182, 96)
Processed: Subject02_2.edf
Saved as: Subject02_2-processed.csv
Shape: (62, 96)
Processed: Subject03_1.edf
Saved as: Subject03_1-processed.csv
Shape: (182, 96)
Processed: Subject03_2.edf
Saved as: Subject03_2-processed.csv
Shape: (62, 96)
Processed: Subject04_1.edf
Saved as: Subject04_1-processed.csv
Shape: (170, 96)
Processed: Subject04_2.edf
Saved as: Subject04_2